## Wheel wavelet decomposition

In [1]:
prefix = '/home/ines/repositories/'
prefix = '/Users/ineslaranjeira/Documents/Repositories/'

In [2]:
""" 
IMPORTS
"""
import os
os.environ["JAX_PLATFORM_NAME"] = "cpu"
import numpy as np
import pandas as pd
from one.api import ONE
from matplotlib import pyplot as plt
from scipy import stats

# Get my functions
from functions import idxs_from_files, fast_wavelet_morlet_convolution_parallel, get_speed
one = ONE(mode='remote')

In [7]:
""" 
LOAD DATA AND PARAMETERS
"""
# LOAD DATA

data_path = prefix + 'representation_learning_variability/paper-individuality/data/design_matrices/kcenia/'
data_path = '/Users/ineslaranjeira/Google Drive/O meu disco/CCU/PhD Project/paper-individuality/data/newly_generated/segmentation/design_matrices/training/'

results_path = prefix + 'representation_learning_variability/paper-individuality/data/wheel_wavelets/1_camera_setup/'
results_path = '/Users/ineslaranjeira/Google Drive/O meu disco/CCU/PhD Project/paper-individuality/data/newly_generated/segmentation/wheel_wavelets/training/'
all_files = os.listdir(data_path)
design_matrices = [item for item in all_files if 'design_matrix' in item and 'standardized' not in item]
idxs, mouse_names = idxs_from_files(design_matrices)

# Wavelet decomposition
f = np.array([.25, .5, 1, 2, 4, 8, 16, 32])
omega0 = 5

## Get training sessions (first session)

In [15]:
# Get my functions
def idxs_from_files(design_matrices):
    
    idxs = []
    mouse_names = []
    for m, mat in enumerate(design_matrices):
        mouse_name = design_matrices[m][51:]
        eid = design_matrices[m][14:50]
        idx = str(eid + '_' + mouse_name)

        if len(idxs) == 0:
            idxs = idx
            mouse_names = mouse_name
        else:
            idxs = np.hstack((idxs, idx))
            mouse_names = np.hstack((mouse_names, mouse_name))
            
    return idxs, mouse_names

data_path = '/Users/ineslaranjeira/Google Drive/O meu disco/CCU/PhD Project/paper-individuality/data/newly_generated/segmentation/design_matrices/training/'
all_files = os.listdir(data_path)
design_matrices = [item for item in all_files if 'design_matrix' in item and 'standardized' not in item]
idxs, mouse_names = idxs_from_files(design_matrices)

learning_data_path = '/Users/ineslaranjeira/Google Drive/O meu disco/CCU/PhD Project/paper-individuality/data/newly_generated/training_data/'
all_files = os.listdir(learning_data_path)
training_mice = []

for m, mouse_name in enumerate(np.unique(mouse_names)):
    
    filename = 'training_data_trials_'+mouse_name
    if filename in all_files:
        training_mice.append(mouse_name)
print(len(training_mice))

# List to store the filtered DataFrames for each mouse
all_filtered_sessions = []
all_filtered_mice = []
for mouse_name in training_mice:
    # 1. Load data
    file_path = f"{learning_data_path}training_data_trials_{mouse_name}"
    try:
        mouse_data = pd.read_parquet(file_path)
    except:
        print(f"Skipping {mouse_name}: File not found.")
        continue
    # 2. Ensure datetime and sort chronologically
    mouse_data['session_date'] = pd.to_datetime(mouse_data['session_date'])
    mouse_data = mouse_data.sort_values('session_date')
    # 3. Define the 3-day window
    start_date = mouse_data['session_date'].min()
    # threshold = start_date + pd.Timedelta(days=3)
    # 4. Filter: Keep all sessions within the window
    # This keeps every trial/row that occurred in those 3 days
    mask = mouse_data['session_date'] == start_date
    filtered_mouse_data = mouse_data.loc[mask].copy()
    mouse_sessions = filtered_mouse_data.session.unique()
    # mouse_sessions = mouse_data.session.unique()
    all_filtered_sessions.extend(mouse_sessions)
    all_filtered_mice.extend([mouse_name])

sessions_to_process = all_filtered_sessions

85


# Wavelets

## why only 85 sessions?


In [18]:
concatenated_subsampled = np.array([])
var = ['avg_wheel_vel']

for m, mat in enumerate(sessions_to_process):

    file_path = one.eid2path(mat)
    if prefix == '/home/ines/repositories/':
        mouse_name = file_path.parts[8]
    else:
        mouse_name = file_path.parts[7]

    session = mat
    filename = data_path + "design_matrix_" + str(session) + '_'  + mouse_name
    design_matrix = pd.read_parquet(filename)
    
    array = np.array(design_matrix[var[0]]) 
    not_nan = ~np.isnan(array)
    clean_array = array[not_nan]# np.array(stats.zscore(design_matrix[var][not_nan_x]))
    
    # Wavelet decomposition of paw position
    dt = np.round(np.mean(np.diff(design_matrix['Bin'])), 3)
    amp, Q, x_hat = fast_wavelet_morlet_convolution_parallel(clean_array, f, omega0, dt)

    # Wavelet transforms
    for i, frequency in enumerate(f):
        # Create new column with frequency
        design_matrix[str(var[0]+str(frequency))] = np.nan
        design_matrix[str(var[0]+str(frequency))][not_nan] = amp[i, :]

    """ SAVE DATA """       
    # Save wavelets
    subname = "wheel_vel_wavelets_"
    filename = results_path + subname + str(session) + '_'  + mouse_name
    design_matrix.to_parquet(filename, compression='gzip')  
    print(mat)

/Users/ineslaranjeira/Documents/Repositories/representation_learning_variability/paper-individuality/0_pre-processing/1_camera_setup/functions.py:649: ComplexWarning: Casting complex values to real discards the imaginary part
  Q[i, :] = q
/var/folders/nt/d2j3zp9d1xzb8wgfrw81j0c40000gn/T/ipykernel_22096/3143546978.py:28: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveat

b117ed10-6871-42b3-9193-ca708dac4353


/Users/ineslaranjeira/Documents/Repositories/representation_learning_variability/paper-individuality/0_pre-processing/1_camera_setup/functions.py:649: ComplexWarning: Casting complex values to real discards the imaginary part
  Q[i, :] = q
/var/folders/nt/d2j3zp9d1xzb8wgfrw81j0c40000gn/T/ipykernel_22096/3143546978.py:28: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveat

aa716244-94e7-45e4-98b8-4ba40430bff0


/Users/ineslaranjeira/Documents/Repositories/representation_learning_variability/paper-individuality/0_pre-processing/1_camera_setup/functions.py:649: ComplexWarning: Casting complex values to real discards the imaginary part
  Q[i, :] = q
/var/folders/nt/d2j3zp9d1xzb8wgfrw81j0c40000gn/T/ipykernel_22096/3143546978.py:28: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveat

...

/Users/ineslaranjeira/Documents/Repositories/representation_learning_variability/paper-individuality/0_pre-processing/1_camera_setup/functions.py:649: ComplexWarning: Casting complex values to real discards the imaginary part
  Q[i, :] = q
/var/folders/nt/d2j3zp9d1xzb8wgfrw81j0c40000gn/T/ipykernel_22096/3143546978.py:28: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveat

378cfd07-0ba1-4c01-9807-28585224cd9c


/Users/ineslaranjeira/Documents/Repositories/representation_learning_variability/paper-individuality/0_pre-processing/1_camera_setup/functions.py:649: ComplexWarning: Casting complex values to real discards the imaginary part
  Q[i, :] = q
/var/folders/nt/d2j3zp9d1xzb8wgfrw81j0c40000gn/T/ipykernel_22096/3143546978.py:28: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveat

d5d4e946-8d4b-4c35-9073-29966729491c


ArrowInvalid: Could not open Parquet input source '<Buffer>': Parquet file size is 0 bytes

In [25]:
pd.read_parquet(filename)

ArrowInvalid: Could not open Parquet input source '<Buffer>': Parquet file size is 0 bytes

In [26]:
filename

'/Users/ineslaranjeira/Google Drive/O meu disco/CCU/PhD Project/paper-individuality/data/newly_generated/segmentation/design_matrices/training/design_matrix_167f98be-6704-430f-b531-2d4ab2aae932_NR_0027'